# Bowen thesis — Appendix C, literal PDROD kernel (Stage 13)
**Scope:** Direct translation of the **published Appendix C excerpt, printed thesis pp. 89–91**. It is **not** a claim to reproduce the unpublished complete Chapter 3 solver.

Important source-fidelity choices:
- Steel point `i` interacts with virtual concrete point `cnode` interpolated from 8 concrete nodes using `N1...N8`.
- `idist`, `nlength`, `oldnlength`, `sssss`, `s1`, `s0`, `sc`, and `bc` follow the printed code.
- Scalar force is multiplied by the source's `vol` **once**, then distributed `-N_a` to the 8 neighbors.
- The source excerpt updates `failfail` **after** forces are computed, and **does not consult it** in the force law. Literal mode replicates this potentially reversible behavior.
- An optional `irreversible_corrected` mode implements a **clearly nonliteral modification**.
- Source does not supply the arrays `hhh`, `nodenode`, `node`, `coordcoordcoord`, `N1...N8`, `bcc`, `s01`, `sc1`, or `vol` values for the benchmark. You must provide these from the actual Chapter 3 setup. We **do not invent those values**.
- The full snippet zeroes `pforce33(i,:)` for all `i = 1...totnode1`; it does not show global force-array ownership. Here we create a separate total-forces array with explicit concrete and steel indexing, preserving its pairwise force contribution. This avoids aliasing unknown global node numbering.


In [ ]:
import numpy as np
from dataclasses import dataclass

@dataclass(frozen=True)
class AppendixCParameters:
    bcc: float
    s01: float
    sc1: float
    delta: float
    vol: float

class AppendixCPublishedKernel:
    """Vectorized form of Appendix C using explicit 0-based steel/concrete indexing."""
    def __init__(self, steel_xyz, concrete_xyz, steel_id, concrete8_id,
                 weights8, virtual_xyz, params: AppendixCParameters):
        self.xs = np.asarray(steel_xyz, float)
        self.xc = np.asarray(concrete_xyz, float)
        self.st = np.asarray(steel_id, np.int64)
        self.idx = np.asarray(concrete8_id, np.int64)
        self.w = np.asarray(weights8, float)
        self.xv = np.asarray(virtual_xyz, float)
        self.p = params
        if self.idx.shape != (len(self.st), 8) or self.w.shape != self.idx.shape:
            raise ValueError('Each coupling entry must have exactly 8 concrete indices and weights')
        if self.xv.shape != (len(self.st), 3): raise ValueError('virtual_xyz must have shape (K,3)')
        if np.any(self.st<0) or np.any(self.st>=len(self.xs)) or np.any(self.idx<0) or np.any(self.idx>=len(self.xc)):
            raise IndexError('Input connectivity is outside the respective steel or concrete array')
        if np.max(np.abs(self.w.sum(axis=1)-1))>1e-10: raise ValueError('Shape weights must sum to one')
        self.reference = self.xv-self.xs[self.st]
        self.idist = np.linalg.norm(self.reference, axis=1)
        if np.any(self.idist<=0): raise ValueError('PDROD bonds cannot have zero initial length')
        if not (params.bcc>0 and params.delta>0 and params.vol>0 and 0<params.s01<params.sc1):
            raise ValueError('Invalid PDROD constants')
        if np.any(self.idist>params.delta): raise ValueError('Neighbors must be within delta')
        self.failfail = np.ones(len(self.st), dtype=np.int8)

    def evaluate(self, uc, us, old_uc=None, old_us=None, dt=None,
                 mode='literal', commit=False):
        if mode not in ('literal','irreversible_corrected'):
            raise ValueError('mode must be literal or irreversible_corrected')
        uc=np.asarray(uc,float); us=np.asarray(us,float)
        if uc.shape != self.xc.shape or us.shape != self.xs.shape:
            raise ValueError('Displacements must match geometry shape')
        dispdisp=np.einsum('ka,kad->kd',self.w, uc[self.idx])
        direction=self.reference+dispdisp-us[self.st]
        nlength=np.linalg.norm(direction,axis=1)
        if np.any(nlength<1e-15): raise FloatingPointError('Current interface length is zero')
        sssss=(nlength-self.idist)/self.idist
        if old_uc is not None and old_us is not None and dt is not None:
            if dt<=0: raise ValueError('dt must be positive')
            old_interp=np.einsum('ka,kad->kd',self.w,np.asarray(old_uc)[self.idx])
            oldnlength=np.linalg.norm(self.reference+old_interp-np.asarray(old_us)[self.st],axis=1)
            s1=np.abs((nlength-oldnlength)/self.idist/dt)
        else:
            oldnlength=np.full_like(nlength,np.nan); s1=np.full_like(nlength,np.nan)
        bc=self.p.bcc*(1-(self.idist/self.p.delta)**2)**2
        scalar=np.where(sssss<=self.p.s01,bc*sssss,
                    np.where(sssss<=self.p.sc1,
                        bc*self.p.s01*(self.p.sc1-sssss)/(self.p.sc1-self.p.s01),0.0))
        new_failed=sssss>self.p.sc1
        if mode=='irreversible_corrected':
            scalar=np.where((self.failfail==0)|new_failed,0.,scalar)
        # Appendix C: dforce = bc*(stretch law)*vol*(deformed vector)/nlength
        dforce=(scalar*self.p.vol)[:,None]*(direction/nlength[:,None])
        fs=np.zeros_like(us); fc=np.zeros_like(uc)
        np.add.at(fs,self.st,dforce)
        for a in range(8): np.add.at(fc,self.idx[:,a],-self.w[:,a,None]*dforce)
        # Printed code computes failure flag after dforce and does not use it above.
        if commit: self.failfail[new_failed]=0
        dmgpar1=float(np.sum(self.failfail*self.p.vol))
        dmgpar2=float(len(self.failfail)*self.p.vol)
        return dict(concrete_force=fc,steel_force=fs,stretch=sssss,stretch_rate=s1,
                    oldnlength=oldnlength,current_length=nlength,dforce=dforce,
                    new_failed=new_failed,failfail=self.failfail.copy(),
                    sample_damage=(1-dmgpar1/dmgpar2 if dmgpar2 else np.nan))
    def reset(self): self.failfail[:]=1


## Source-to-Python correspondence

| Appendix C variable or operation | Python | Fidelity |
|---|---|---|
| `nodenode(i,j)` → `node(cnode,1:8)` | `steel_id`, `concrete8_id` | Same connectivity concept; requires source arrays |
| `N1...N8` interpolation of `disp`, `olddisp` | `einsum` over weights | Same operation |
| `idist`, `nlength`, `oldnlength` | vector norms | Same operation |
| `sssss` and `s1` | stretch and absolute stretch-rate | Same operation |
| `s0=s01`, `sc=sc1` | `params.s01`, `params.sc1` | Same operation, **no rate factor applied in Appendix C** |
| `bc=bcc*(1-(idist/delta)**2)**2` | `bc` array | Same operation |
| `dforce1:3` multiplied by `vol` | `dforce` | Same operation |
| `pforce33(i,:) += dforce` | `steel_force` | Same per-interaction contribution, separated arrays |
| `pforce33(cnodea,:) -= Na*dforce` | `concrete_force` | Same per-interaction contribution |
| `failfail(i,j)=0` if `stretch>sc` | `commit=True` state update | Same post-force update |
| force omitted when `failfail=0` | Optional corrected mode **only** | **Not in printed excerpt** |

**Source ambiguity:** the excerpt computes `s1` but does not use it to modify `s0`, `sc`, or `bc` in Appendix C. No extra rate-dependent PDROD adjustment is introduced in literal mode.


## Controlled kernel tests — artificial geometry is for verification only

The following small configurations are **unit tests**, not Bowen beam parameters or predictions. The numerical constants are deliberately arbitrary and isolated from the published model.

In [ ]:
# One steel node at the origin and one virtual concrete point, represented by 8 nodes.
steel=np.array([[0.,0.,0.]])
concrete=np.array([[0.010,dy,dz] for dy in (-0.002,0.002) for dz in (-0.002,0.002)] +
                  [[0.012,dy,dz] for dy in (-0.002,0.002) for dz in (-0.002,0.002)])
weights=np.full((1,8),1/8)
virtual=(weights@concrete).reshape(1,3)
p_test=AppendixCParameters(bcc=2e8,s01=1e-3,sc1=4e-3,delta=0.02,vol=1e-6)
ker=AppendixCPublishedKernel(steel,concrete,[0],[np.arange(8)],weights,virtual,p_test)
uc0=np.zeros_like(concrete);us0=np.zeros_like(steel)
r0=ker.evaluate(uc0,us0)
assert np.max(abs(r0['dforce']))==0
assert np.isclose(ker.idist[0],0.011)
print('PASS zero deformation, distance, and reference interpolation')
# Move virtual concrete point by 11 micrometers => tensile stretch 1e-3.
uc1=uc0.copy();uc1[:,0]=11e-6
r1=ker.evaluate(uc1,us0,old_uc=uc0,old_us=us0,dt=1e-5)
expected_bc=p_test.bcc*(1-(ker.idist[0]/p_test.delta)**2)**2
expected_force=expected_bc*p_test.s01*p_test.vol
np.testing.assert_allclose(r1['steel_force'][0,0],expected_force,rtol=1e-12)
np.testing.assert_allclose(r1['concrete_force'].sum(axis=0)+r1['steel_force'].sum(axis=0),0,atol=1e-12)
np.testing.assert_allclose(r1['stretch_rate'][0],100.,rtol=1e-12)
print('PASS source force, 8-node distribution, force balance, and stretch rate')
# Test linear unloading into compression; Appendix C uses same linear branch.
uc_comp=uc0.copy();uc_comp[:,0]=-1.1e-6
r_comp=ker.evaluate(uc_comp,us0)
assert r_comp['steel_force'][0,0]<0
print('PASS negative compression-side force')


In [ ]:
# Crucial literal-vs-corrected history test: printed source permits later reloading.
uc_break=uc0.copy();uc_break[:,0]=ker.idist[0]*0.005
rb=ker.evaluate(uc_break,us0,mode='literal',commit=True)
assert ker.failfail[0]==0 and np.allclose(rb['dforce'],0)
uc_reload=uc0.copy();uc_reload[:,0]=ker.idist[0]*0.0005
literal=ker.evaluate(uc_reload,us0,mode='literal')['steel_force'][0,0]
corrected=ker.evaluate(uc_reload,us0,mode='irreversible_corrected')['steel_force'][0,0]
assert literal>0 and corrected==0
print('PASS source fidelity distinction: literal restored force:',literal,'N; corrected:',corrected,'N')
ker.reset()


## Integrating into Stage 12 (requires original benchmark inputs)

Stage 12's `FastInterface` is **not** the Appendix C literal version because it enforces broken-state suppression during force computation. To replace it, provide the actual steel and concrete coordinate arrays, interpolation connectivity and original `bcc`, `s01`, `sc1`, `delta`, `vol` values.

```python
pdrod = AppendixCPublishedKernel(
    steel_xyz, xyz, sample_steel, sample_idx, sample_w, sample_pos,
    AppendixCParameters(bcc=original_bcc, s01=original_s01,
                        sc1=original_sc1, delta=original_delta, vol=original_vol))

# In the dynamic force assembly:
res = pdrod.evaluate(uc, us, old_uc=old_uc, old_us=old_us,
                     dt=dt, mode='literal', commit=True)
Fc += res['concrete_force']
Fs += res['steel_force']
```

`old_uc` and `old_us` must be snapshots of the *previous* dynamic step, not references to arrays mutated in place. This standalone kernel is for comparison and numerical auditing; an accurate dynamic integration still needs the complete original update sequence and global force indexing.
